# Omine Laughter Segmentation

This notebook runs the Omine et al. Interspeech 2024 laughter segmentation model on an audio file and exports detected laughter timestamps.

Sources:
- Paper: https://www.isca-archive.org/interspeech_2024/omine24_interspeech.html
- GitHub implementation: https://github.com/omine-me/LaughterSegmentation
- Model weights: https://huggingface.co/omine-me/LaughterSegmentation

Notes:
- The public model file is about 1.26 GB and is described by the authors as research-use only.
- The first run also downloads the underlying wav2vec2 model used by the implementation.
- For Colab, use Runtime > Change runtime type > GPU if available.


In [ ]:
#@title Environment and repository setup

import os
import shutil
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

WORK_ROOT = Path('/content') if IN_COLAB else Path.cwd()
REPO_DIR = (WORK_ROOT / 'LaughterSegmentation').resolve()

def run_command(command, cwd=None):
    print('+', ' '.join(map(str, command)))
    subprocess.check_call(list(map(str, command)), cwd=str(cwd) if cwd else None)

if IN_COLAB and not shutil.which('ffmpeg'):
    run_command(['apt-get', 'update', '-qq'])
    run_command(['apt-get', 'install', '-y', '-qq', 'ffmpeg'])

if not REPO_DIR.exists():
    run_command(['git', 'clone', 'https://github.com/omine-me/LaughterSegmentation.git', REPO_DIR])
else:
    print(f'Using existing repository: {REPO_DIR}')

for package_dir in [REPO_DIR / 'train', REPO_DIR / 'evaluation', REPO_DIR / 'evaluation' / '_utils']:
    package_dir.mkdir(parents=True, exist_ok=True)
    (package_dir / '__init__.py').touch(exist_ok=True)

repo_path = str(REPO_DIR)
train_path = str(REPO_DIR / 'train')
sys.path = [entry for entry in sys.path if entry != train_path]
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

print(f'IN_COLAB: {IN_COLAB}')
print(f'REPO_DIR: {REPO_DIR}')


In [ ]:
#@title Install inference dependencies

import importlib.util
import subprocess
import sys

def pip_install(packages):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *packages])

base_packages = [
    'librosa>=0.10.1',
    'soundfile>=0.12.1',
    'pydub>=0.25.1',
    'safetensors>=0.4.1',
    'scipy>=1.10.0',
    'transformers==4.36.1',
    'huggingface_hub>=0.20.0',
    'pandas>=1.5.0',
    'matplotlib>=3.6.0',
    'tqdm>=4.64.0',
]

pip_install(base_packages)

if importlib.util.find_spec('torch') is None:
    pip_install(['torch'])

print('Dependencies installed.')


In [ ]:
#@title Download Omine model weights from Hugging Face

from pathlib import Path
from huggingface_hub import hf_hub_download

HF_MODEL_REPO = 'omine-me/LaughterSegmentation'
MODEL_DIR = REPO_DIR / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = Path(
    hf_hub_download(
        repo_id=HF_MODEL_REPO,
        filename='model.safetensors',
        local_dir=str(MODEL_DIR),
    )
).resolve()

print(f'Model path: {MODEL_PATH}')
print(f'Model size: {MODEL_PATH.stat().st_size / (1024 ** 3):.2f} GB')


In [ ]:
#@title Optional Colab upload helper

if IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
    for name in uploaded:
        print(f'Uploaded: /content/{name}')
else:
    print('This helper is only for Google Colab. Set AUDIO_FILE directly in the next cell when running locally.')


In [ ]:
#@title Choose audio file and Drive output folder

from pathlib import Path

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

# In Colab, upload your file first or point this at a file in Google Drive.
# Locally, use an absolute path or a path relative to the notebook folder.
AUDIO_FILE = '/content/your_audio.mp3'  #@param {type:"string"}
OUTPUT_ROOT = '/content/drive/MyDrive/FreekdeJonge/omine_laughter_output'  #@param {type:"string"}

# Omine defaults from the published implementation.
INPUT_SEC = 7  #@param {type:"integer"}
BATCH_SIZE = 10  #@param {type:"integer"}

AUDIO_PATH = Path(AUDIO_FILE).expanduser().resolve()
if not AUDIO_PATH.exists():
    raise FileNotFoundError(f'Audio file not found: {AUDIO_PATH}')

OUTPUT_ROOT_PATH = Path(OUTPUT_ROOT).expanduser().resolve()
OUTPUT_DIR = (OUTPUT_ROOT_PATH / AUDIO_PATH.stem).resolve()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Audio: {AUDIO_PATH}')
print(f'Output root: {OUTPUT_ROOT_PATH}')
print(f'Output: {OUTPUT_DIR}')
print(f'Model: {MODEL_PATH}')


In [ ]:
#@title Optional transcript file for text alignment

from pathlib import Path

# Upload the transcript to /content with the upload helper, or point to a Drive file.
# Supported: JSON transcripts with lines/segments lists, or block-style TXT transcripts.
TRANSCRIPT_FILE = ''  #@param {type:"string"}
TRANSCRIPT_FPS = 25.0  #@param {type:"number"}
CONTEXT_LOOKBACK_SECONDS = 60.0  #@param {type:"number"}

WITH_TEXT_PATH = OUTPUT_DIR / f'{AUDIO_PATH.stem}_laughter_with_text.json'
WITH_TEXT_CSV = OUTPUT_DIR / f'{AUDIO_PATH.stem}_laughter_with_text.csv'

TRANSCRIPT_PATH = Path(TRANSCRIPT_FILE).expanduser().resolve() if TRANSCRIPT_FILE.strip() else None

if TRANSCRIPT_PATH is None:
    print('No transcript configured yet. Set TRANSCRIPT_FILE when you want to align laughter to text.')
elif not TRANSCRIPT_PATH.exists():
    raise FileNotFoundError(f'Transcript file not found: {TRANSCRIPT_PATH}')
else:
    print(f'Transcript: {TRANSCRIPT_PATH}')
    print(f'Output with text JSON: {WITH_TEXT_PATH}')
    print(f'Output with text CSV: {WITH_TEXT_CSV}')


In [ ]:
#@title Run Omine laughter segmentation

import json
import os
import sys
from pathlib import Path

# The upstream inference.py expects this submodule to be importable as safetensors.torch.
import safetensors.torch  # noqa: F401

os.chdir(REPO_DIR)

# The upstream repo has train/model.py but may not include __init__.py files.
# Colab can also have a stale top-level `train` module, so make Omine's train/
# folder an explicit package and remove any shadowing module before import.
for package_dir in [REPO_DIR / 'train', REPO_DIR / 'evaluation', REPO_DIR / 'evaluation' / '_utils']:
    package_dir.mkdir(parents=True, exist_ok=True)
    (package_dir / '__init__.py').touch(exist_ok=True)

repo_path = str(REPO_DIR)
train_path = str(REPO_DIR / 'train')
sys.path = [entry for entry in sys.path if entry != train_path]
if repo_path in sys.path:
    sys.path.insert(0, sys.path.pop(sys.path.index(repo_path)))
else:
    sys.path.insert(0, repo_path)

train_module = sys.modules.get('train')
if train_module is not None and not hasattr(train_module, '__path__'):
    del sys.modules['train']
sys.modules.pop('inference', None)

from inference import main as run_omine_inference

run_omine_inference(
    audio_path=str(AUDIO_PATH),
    output_dir=str(OUTPUT_DIR),
    model_path=str(MODEL_PATH),
    input_sec=INPUT_SEC,
    batch_size=BATCH_SIZE,
)

RAW_OUTPUT_JSON = OUTPUT_DIR / f'{AUDIO_PATH.stem}.json'
if not RAW_OUTPUT_JSON.exists():
    raise FileNotFoundError(f'Expected output JSON was not created: {RAW_OUTPUT_JSON}')

print(f'Saved raw Omine output: {RAW_OUTPUT_JSON}')


In [ ]:
#@title Inspect and export timestamp table

import json
from pathlib import Path

import pandas as pd
from IPython.display import display

with RAW_OUTPUT_JSON.open('r', encoding='utf-8') as f:
    raw_events = json.load(f)

segments = []
for key, event in raw_events.items():
    start = float(event['start_sec'])
    end = float(event['end_sec'])
    segments.append(
        {
            'index': int(key),
            'start_sec': start,
            'end_sec': end,
            'duration_sec': end - start,
        }
    )

df = pd.DataFrame(segments).sort_values('start_sec').reset_index(drop=True)
if not df.empty:
    df['index'] = range(len(df))

NORMALIZED_JSON = OUTPUT_DIR / f'{AUDIO_PATH.stem}_laughter_segments.json'
CSV_PATH = OUTPUT_DIR / f'{AUDIO_PATH.stem}_laughter_segments.csv'

payload = {
    'source_audio': str(AUDIO_PATH),
    'model': 'omine-me/LaughterSegmentation',
    'raw_output_json': str(RAW_OUTPUT_JSON),
    'segment_count': int(len(df)),
    'total_laughter_seconds': float(df['duration_sec'].sum()) if not df.empty else 0.0,
    'segments': df.to_dict(orient='records'),
}

NORMALIZED_JSON.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding='utf-8')
df.to_csv(CSV_PATH, index=False)

print(f'Detected segments: {len(df)}')
print(f'Total laughter seconds: {payload["total_laughter_seconds"]:.2f}')
print(f'Normalized JSON: {NORMALIZED_JSON}')
print(f'CSV: {CSV_PATH}')
display(df.head(30))


In [ ]:
#@title Align laughter timestamps with transcript text

import json
import re
from pathlib import Path

import pandas as pd
from IPython.display import display

TRIGGER_LOOKBACK_SECONDS = 15.0  #@param {type:"number"}

def parse_timecode(tc: str, fps: float) -> float:
    tc = tc.strip().replace(',', '.')
    parts = tc.split(':')
    if len(parts) == 4:
        hh, mm, ss, ff = parts
        return int(hh) * 3600 + int(mm) * 60 + int(ss) + (int(ff) / fps)
    if len(parts) == 3:
        hh, mm, ss = parts
        return int(hh) * 3600 + int(mm) * 60 + float(ss)
    raise ValueError(f'Invalid timecode: {tc}')

def parse_time_value(value, fps: float) -> float:
    if isinstance(value, (int, float)):
        return float(value)
    text = str(value).strip()
    try:
        return float(text)
    except ValueError:
        return parse_timecode(text, fps)

def parse_time_range(line: str, fps: float):
    match = re.match(r'\s*(\S+)\s*-\s*(\S+)\s*', line)
    if not match:
        raise ValueError(f'Invalid time range line: {line}')
    return parse_timecode(match.group(1), fps), parse_timecode(match.group(2), fps)

def normalize_text(text: str) -> str:
    return re.sub(r'\s+', ' ', str(text)).strip()

def should_exclude_text(text: str) -> bool:
    normalized = normalize_text(text).upper()
    if not normalized:
        return False
    return any(marker in normalized for marker in ('GELACH', 'APPLAUS', 'MUZIEK', 'LAUGHTER', 'APPLAUSE', 'MUSIC'))

def seconds_to_timecode(seconds: float, fps: float = 25.0) -> str:
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    f = min(round((seconds - int(seconds)) * fps), int(fps) - 1)
    return f'{h:02d}:{m:02d}:{s:02d}:{f:02d}'

def parse_text_transcript(raw: str, fps: float):
    blocks = [block for block in re.split(r'\n\s*\n', raw.strip()) if block.strip()]
    segments = []
    for block in blocks:
        lines = [line.strip() for line in block.splitlines() if line.strip()]
        if len(lines) < 2:
            continue
        start, end = parse_time_range(lines[0], fps)
        speaker = normalize_text(lines[1])
        text = normalize_text(' '.join(lines[2:])) if len(lines) > 2 else ''
        segments.append({'start': start, 'end': end, 'speaker': speaker, 'text': text})
    return segments

def first_present(entry, keys):
    for key in keys:
        if key in entry and entry[key] is not None:
            return entry[key]
    return None

def parse_json_transcript(data, fps: float):
    if isinstance(data, dict):
        line_items = data.get('lines') or data.get('segments') or data.get('results') or []
    else:
        line_items = data
    if not isinstance(line_items, list):
        raise ValueError('JSON transcript must be a list or contain lines/segments/results.')

    segments = []
    for entry in line_items:
        if not isinstance(entry, dict):
            continue
        start_value = first_present(entry, ['startTime', 'start', 'start_sec', 'start_seconds', 'begin'])
        end_value = first_present(entry, ['endTime', 'end', 'end_sec', 'end_seconds', 'finish'])
        if start_value is None or end_value is None:
            continue
        speaker = normalize_text(first_present(entry, ['speakerDesignation', 'speaker', 'speaker_label']) or '')
        text = normalize_text(first_present(entry, ['text', 'transcript', 'content']) or '')
        segments.append(
            {
                'start': parse_time_value(start_value, fps),
                'end': parse_time_value(end_value, fps),
                'speaker': speaker,
                'text': text,
            }
        )
    return segments

def parse_transcript(path: Path, fps: float):
    raw = path.read_text(encoding='utf-8')
    stripped = raw.lstrip()
    if stripped.startswith('{') or stripped.startswith('['):
        return parse_json_transcript(json.loads(raw), fps)
    return parse_text_transcript(raw, fps)

def join_segment_text(segments):
    return normalize_text(' '.join(s['text'] for s in segments if s.get('text') and not should_exclude_text(s['text'])))

def unique_speakers(segments):
    speakers = []
    for segment in segments:
        speaker = segment.get('speaker') or ''
        if speaker and speaker not in speakers:
            speakers.append(speaker)
    return speakers

def align_laughter_rows_to_transcript(laughter_df, transcript_segments, fps=25.0):
    results = []
    transcript_segments = sorted(transcript_segments, key=lambda item: (item['start'], item['end']))
    previous_laughter_end = 0.0

    for item in laughter_df.to_dict(orient='records'):
        start = float(item['start_sec'])
        end = float(item['end_sec'])
        overlaps = [s for s in transcript_segments if start <= s['end'] and end >= s['start']]
        trigger_start = max(0.0, start - TRIGGER_LOOKBACK_SECONDS)
        trigger_segments = overlaps or [
            s for s in transcript_segments
            if s['text'] and not should_exclude_text(s['text']) and s['end'] > trigger_start and s['end'] <= start
        ]

        current_text_start = overlaps[0]['start'] if overlaps else start
        short_context_segments = [
            s for s in transcript_segments
            if s['text'] and not should_exclude_text(s['text']) and s['start'] >= previous_laughter_end and s['end'] <= current_text_start
        ]
        long_context_start = max(0.0, current_text_start - CONTEXT_LOOKBACK_SECONDS)
        long_context_segments = [
            s for s in transcript_segments
            if s['text'] and not should_exclude_text(s['text']) and s['end'] > long_context_start and s['start'] < current_text_start
        ]

        results.append(
            {
                'index': int(item['index']),
                'start_sec': start,
                'end_sec': end,
                'duration_sec': float(item.get('duration_sec', end - start)),
                'time_range': f'{seconds_to_timecode(start, fps)} - {seconds_to_timecode(end, fps)}',
                'clip_filename': item.get('clip_filename'),
                'clip_path': item.get('clip_path'),
                'text': join_segment_text(overlaps),
                'trigger_text': join_segment_text(trigger_segments),
                'context': join_segment_text(short_context_segments),
                'context_long': join_segment_text(long_context_segments),
                'speakers': unique_speakers(overlaps + trigger_segments),
            }
        )
        previous_laughter_end = max(previous_laughter_end, end)

    return results

if TRANSCRIPT_PATH is None:
    print('Skipping transcript alignment: set TRANSCRIPT_FILE in the transcript config cell first.')
else:
    transcript_segments = parse_transcript(TRANSCRIPT_PATH, TRANSCRIPT_FPS)
    if not transcript_segments:
        raise ValueError(f'No transcript segments parsed from {TRANSCRIPT_PATH}')

    aligned_segments = align_laughter_rows_to_transcript(df, transcript_segments, fps=TRANSCRIPT_FPS)
    df_with_text = pd.DataFrame(aligned_segments)
    df_with_text.to_csv(WITH_TEXT_CSV, index=False)

    with_text_payload = {
        'source_audio': str(AUDIO_PATH),
        'transcript': str(TRANSCRIPT_PATH),
        'transcript_segment_count': len(transcript_segments),
        'laughter_segment_count': len(aligned_segments),
        'segments': aligned_segments,
    }
    WITH_TEXT_PATH.write_text(json.dumps(with_text_payload, indent=2, ensure_ascii=False), encoding='utf-8')

    print(f'Loaded {len(transcript_segments)} transcript segment(s) from {TRANSCRIPT_PATH.name}')
    print(f'Saved transcript-aligned JSON: {WITH_TEXT_PATH}')
    print(f'Saved transcript-aligned CSV: {WITH_TEXT_CSV}')
    display(df_with_text.head(30))


In [ ]:
#@title Export Omine laughter clips to Drive

import json
import subprocess

import pandas as pd
from IPython.display import display

CLIP_PADDING_SECONDS = 0.10  #@param {type:"number"}
CLIP_SAMPLE_RATE = 16000  #@param {type:"integer"}
OVERWRITE_LAUGHTER_CLIPS = True  #@param {type:"boolean"}

CLIP_DIR = OUTPUT_DIR / f'{AUDIO_PATH.stem}_laughter_clips'
CLIP_DIR.mkdir(parents=True, exist_ok=True)

clip_records = []

for segment in df.to_dict(orient='records'):
    segment_index = int(segment['index'])
    start_sec = max(0.0, float(segment['start_sec']) - CLIP_PADDING_SECONDS)
    end_sec = float(segment['end_sec']) + CLIP_PADDING_SECONDS
    duration_sec = max(0.01, end_sec - start_sec)
    clip_path = CLIP_DIR / f'laugh_{segment_index:03d}_{float(segment["start_sec"]):.2f}-{float(segment["end_sec"]):.2f}.wav'

    if OVERWRITE_LAUGHTER_CLIPS or not clip_path.exists():
        subprocess.run(
            [
                'ffmpeg',
                '-y',
                '-hide_banner',
                '-loglevel',
                'error',
                '-ss',
                f'{start_sec:.3f}',
                '-t',
                f'{duration_sec:.3f}',
                '-i',
                str(AUDIO_PATH),
                '-ar',
                str(CLIP_SAMPLE_RATE),
                '-ac',
                '1',
                str(clip_path),
            ],
            check=True,
        )

    clip_records.append(
        {
            'index': segment_index,
            'clip_path': str(clip_path),
            'clip_filename': clip_path.name,
        }
    )

clip_df = pd.DataFrame(clip_records)
if not clip_df.empty:
    df = df.drop(columns=['clip_path', 'clip_filename'], errors='ignore').merge(clip_df, on='index', how='left')

df.to_csv(CSV_PATH, index=False)
updated_payload = {
    'source_audio': str(AUDIO_PATH),
    'model': 'omine-me/LaughterSegmentation',
    'raw_output_json': str(RAW_OUTPUT_JSON),
    'clip_dir': str(CLIP_DIR),
    'clip_padding_seconds': CLIP_PADDING_SECONDS,
    'clip_sample_rate': CLIP_SAMPLE_RATE,
    'segment_count': int(len(df)),
    'total_laughter_seconds': float(df['duration_sec'].sum()) if not df.empty else 0.0,
    'segments': df.to_dict(orient='records'),
}
NORMALIZED_JSON.write_text(json.dumps(updated_payload, indent=2, ensure_ascii=False), encoding='utf-8')

if 'df_with_text' in globals() and not clip_df.empty:
    df_with_text = df_with_text.drop(columns=['clip_path', 'clip_filename'], errors='ignore').merge(clip_df, on='index', how='left')
    df_with_text.to_csv(WITH_TEXT_CSV, index=False)
    if WITH_TEXT_PATH.exists():
        with_text_payload = json.loads(WITH_TEXT_PATH.read_text(encoding='utf-8'))
    else:
        with_text_payload = {'source_audio': str(AUDIO_PATH), 'transcript': str(TRANSCRIPT_PATH) if TRANSCRIPT_PATH else None}
    with_text_payload['clip_dir'] = str(CLIP_DIR)
    with_text_payload['segments'] = df_with_text.to_dict(orient='records')
    WITH_TEXT_PATH.write_text(json.dumps(with_text_payload, indent=2, ensure_ascii=False), encoding='utf-8')


CLIPS_MANIFEST = OUTPUT_DIR / f'{AUDIO_PATH.stem}_laughter_clips_manifest.json'
CLIPS_MANIFEST.write_text(
    json.dumps(
        {
            'source_audio': str(AUDIO_PATH),
            'clip_dir': str(CLIP_DIR),
            'clip_count': len(clip_records),
            'clips': clip_records,
        },
        indent=2,
        ensure_ascii=False,
    ),
    encoding='utf-8',
)

print(f'Exported {len(clip_records)} laughter clip(s) to Drive: {CLIP_DIR}')
print(f'Updated CSV with clip paths: {CSV_PATH}')
print(f'Updated JSON with clip paths: {NORMALIZED_JSON}')
print(f'Clips manifest: {CLIPS_MANIFEST}')
display(df.head(30))


In [ ]:
#@title Build tagger verification windows from Omine detections

import json
from pathlib import Path

import librosa
import pandas as pd
from IPython.display import display

# AudioSet taggers are usually trained/evaluated on 10-second clips.
TAGGER_WINDOW_SECONDS = 10.0  #@param {type:"number"}
FALLBACK_SLIDE_SECONDS = 5.0  #@param {type:"number"}

try:
    audio_duration = librosa.get_duration(path=str(AUDIO_PATH))
except TypeError:
    audio_duration = librosa.get_duration(filename=str(AUDIO_PATH))

def centered_window(start_sec, end_sec, total_duration, window_seconds):
    center = (float(start_sec) + float(end_sec)) / 2.0
    window_start = max(0.0, center - window_seconds / 2.0)
    window_end = min(float(total_duration), window_start + window_seconds)
    window_start = max(0.0, window_end - window_seconds)
    return window_start, window_end

windows = []
if not df.empty:
    for row in df.itertuples():
        window_start, window_end = centered_window(
            row.start_sec,
            row.end_sec,
            audio_duration,
            TAGGER_WINDOW_SECONDS,
        )
        windows.append(
            {
                'window_index': len(windows),
                'window_start_sec': round(window_start, 3),
                'window_end_sec': round(window_end, 3),
                'window_duration_sec': round(window_end - window_start, 3),
                'source': 'omine_centered_window',
                'omine_segment_index': int(row.index),
                'omine_start_sec': float(row.start_sec),
                'omine_end_sec': float(row.end_sec),
                'omine_duration_sec': float(row.duration_sec),
            }
        )
else:
    start = 0.0
    while start < audio_duration:
        end = min(audio_duration, start + TAGGER_WINDOW_SECONDS)
        windows.append(
            {
                'window_index': len(windows),
                'window_start_sec': round(start, 3),
                'window_end_sec': round(end, 3),
                'window_duration_sec': round(end - start, 3),
                'source': 'fallback_whole_file_sliding_window',
                'omine_segment_index': None,
                'omine_start_sec': None,
                'omine_end_sec': None,
                'omine_duration_sec': None,
            }
        )
        start += FALLBACK_SLIDE_SECONDS

verification_windows_df = pd.DataFrame(windows)

WINDOWS_CSV = OUTPUT_DIR / f'{AUDIO_PATH.stem}_tagger_verification_windows.csv'
WINDOWS_JSON = OUTPUT_DIR / f'{AUDIO_PATH.stem}_tagger_verification_windows.json'
verification_windows_df.to_csv(WINDOWS_CSV, index=False)
WINDOWS_JSON.write_text(
    json.dumps(
        {
            'source_audio': str(AUDIO_PATH),
            'audio_duration_sec': float(audio_duration),
            'window_seconds': TAGGER_WINDOW_SECONDS,
            'windows': verification_windows_df.to_dict(orient='records'),
        },
        indent=2,
        ensure_ascii=False,
    ),
    encoding='utf-8',
)

print(f'Built {len(verification_windows_df)} verification window(s).')
print(f'Windows CSV: {WINDOWS_CSV}')
display(verification_windows_df.head(30))


In [ ]:
#@title AST AudioSet laughter scores for verification windows

import gc
import json

import librosa
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from tqdm.auto import tqdm
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification

AST_MODEL_ID = 'MIT/ast-finetuned-audioset-10-10-0.4593'  #@param {type:"string"}
AST_SAMPLE_RATE = 16000

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

tagger_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ast_feature_extractor = AutoFeatureExtractor.from_pretrained(AST_MODEL_ID)
ast_model = AutoModelForAudioClassification.from_pretrained(AST_MODEL_ID).to(tagger_device)
ast_model.eval()

def find_label_ids(id2label, exact_label='laughter'):
    exact_ids = []
    related_ids = []
    for raw_idx, raw_label in id2label.items():
        label = str(raw_label).strip().lower()
        idx = int(raw_idx)
        if label == exact_label:
            exact_ids.append(idx)
        elif exact_label in label:
            related_ids.append(idx)
    return exact_ids, related_ids

ast_laughter_ids, ast_related_laughter_ids = find_label_ids(ast_model.config.id2label)
if not ast_laughter_ids:
    ast_laughter_ids = [16]
ast_any_laughter_ids = sorted(set(ast_laughter_ids + ast_related_laughter_ids))

def load_audio_window(audio_path, start_sec, end_sec, sample_rate):
    duration = max(0.01, float(end_sec) - float(start_sec))
    audio, _ = librosa.load(
        str(audio_path),
        sr=sample_rate,
        mono=True,
        offset=float(start_sec),
        duration=duration,
    )
    return audio.astype(np.float32)

ast_rows = []
with torch.no_grad():
    for row in tqdm(verification_windows_df.itertuples(), total=len(verification_windows_df), desc='AST windows'):
        audio = load_audio_window(AUDIO_PATH, row.window_start_sec, row.window_end_sec, AST_SAMPLE_RATE)
        inputs = ast_feature_extractor(audio, sampling_rate=AST_SAMPLE_RATE, return_tensors='pt')
        inputs = {key: value.to(tagger_device) for key, value in inputs.items()}
        logits = ast_model(**inputs).logits[0]
        probs = torch.sigmoid(logits).detach().cpu().numpy()

        laughter_score = float(max(probs[idx] for idx in ast_laughter_ids))
        any_laughter_score = float(max(probs[idx] for idx in ast_any_laughter_ids))
        top_ids = probs.argsort()[-5:][::-1]
        top_labels = [
            {
                'label': ast_model.config.id2label[int(idx)],
                'score': float(probs[int(idx)]),
            }
            for idx in top_ids
        ]

        ast_rows.append(
            {
                'window_index': int(row.window_index),
                'ast_laughter_score': laughter_score,
                'ast_any_laughter_score': any_laughter_score,
                'ast_top_labels': json.dumps(top_labels, ensure_ascii=False),
            }
        )

AST_RESULTS_DF = pd.DataFrame(ast_rows)
AST_RESULTS_CSV = OUTPUT_DIR / f'{AUDIO_PATH.stem}_ast_laughter_scores.csv'
AST_RESULTS_DF.to_csv(AST_RESULTS_CSV, index=False)

print(f'AST laughter label ids: {ast_laughter_ids}')
print(f'AST related laughter ids: {ast_related_laughter_ids}')
print(f'AST scores CSV: {AST_RESULTS_CSV}')
display(AST_RESULTS_DF.head(30))


In [ ]:
#@title Whisper-AT AudioSet laughter scores for verification windows

import importlib.util
import subprocess
import sys

import numpy as np
import pandas as pd
import torch
from IPython.display import display

def pip_install_args(args):
    command = [sys.executable, '-m', 'pip', 'install', '-q', *args]
    print('+', ' '.join(command))
    subprocess.check_call(command)

if importlib.util.find_spec('whisper_at') is None:
    # `pip install whisper-at` can fail in current Colab because the package
    # declares old dependencies such as triton==2.0.0. Install compatible
    # runtime deps first, then install Whisper-AT without resolving deps.
    required_modules = [
        ('numba', 'numba'),
        ('more_itertools', 'more-itertools'),
        ('regex', 'regex'),
        ('requests', 'requests'),
        ('tiktoken', 'tiktoken>=0.7.0'),
    ]
    for module_name, package_name in required_modules:
        if importlib.util.find_spec(module_name) is None:
            pip_install_args([package_name])

    if sys.version_info >= (3, 12):
        pip_install_args(['--upgrade', 'tiktoken>=0.7.0'])

    pip_install_args(['--no-deps', 'whisper-at'])

import whisper_at as whisper

# base is the lightest useful default. Use small/medium/large-v1 for stronger but slower tagging.
WHISPER_AT_MODEL_NAME = 'base'  #@param ['tiny', 'base', 'small', 'medium', 'large-v1'] {allow-input: true}
WHISPER_AT_TIME_RES = 10.0  #@param {type:"number"}
WHISPER_AT_LAUGHTER_ID = 16

whisper_at_model = whisper.load_model(WHISPER_AT_MODEL_NAME)
whisper_at_result = whisper_at_model.transcribe(
    str(AUDIO_PATH),
    at_time_res=WHISPER_AT_TIME_RES,
    fp16=torch.cuda.is_available(),
    verbose=False,
    condition_on_previous_text=False,
)

audio_tag_logits = whisper_at_result['audio_tag']
if isinstance(audio_tag_logits, torch.Tensor):
    audio_tag_logits = audio_tag_logits.detach().cpu()
else:
    audio_tag_logits = torch.tensor(audio_tag_logits)

whisper_laughter_scores = torch.sigmoid(audio_tag_logits[:, WHISPER_AT_LAUGHTER_ID]).numpy()
whisper_bins = pd.DataFrame(
    {
        'bin_index': np.arange(len(whisper_laughter_scores)),
        'bin_start_sec': np.arange(len(whisper_laughter_scores)) * WHISPER_AT_TIME_RES,
        'bin_end_sec': (np.arange(len(whisper_laughter_scores)) + 1) * WHISPER_AT_TIME_RES,
        'whisper_at_laughter_score': whisper_laughter_scores,
    }
)

whisper_rows = []
for row in verification_windows_df.itertuples():
    overlaps = whisper_bins[
        (whisper_bins['bin_start_sec'] < row.window_end_sec)
        & (whisper_bins['bin_end_sec'] > row.window_start_sec)
    ]
    if overlaps.empty:
        max_score = np.nan
        mean_score = np.nan
        overlap_bins = ''
    else:
        max_score = float(overlaps['whisper_at_laughter_score'].max())
        mean_score = float(overlaps['whisper_at_laughter_score'].mean())
        overlap_bins = ','.join(map(str, overlaps['bin_index'].astype(int).tolist()))

    whisper_rows.append(
        {
            'window_index': int(row.window_index),
            'whisper_at_laughter_score_max': max_score,
            'whisper_at_laughter_score_mean': mean_score,
            'whisper_at_overlap_bins': overlap_bins,
        }
    )

WHISPER_AT_RESULTS_DF = pd.DataFrame(whisper_rows)
WHISPER_AT_BINS_CSV = OUTPUT_DIR / f'{AUDIO_PATH.stem}_whisper_at_laughter_bins.csv'
WHISPER_AT_RESULTS_CSV = OUTPUT_DIR / f'{AUDIO_PATH.stem}_whisper_at_laughter_scores.csv'
whisper_bins.to_csv(WHISPER_AT_BINS_CSV, index=False)
WHISPER_AT_RESULTS_DF.to_csv(WHISPER_AT_RESULTS_CSV, index=False)

print(f'Whisper-AT bins CSV: {WHISPER_AT_BINS_CSV}')
print(f'Whisper-AT window scores CSV: {WHISPER_AT_RESULTS_CSV}')
display(WHISPER_AT_RESULTS_DF.head(30))


In [ ]:
#@title Combine Omine, AST, and Whisper-AT review scores

import json

import pandas as pd
from IPython.display import display

AST_REVIEW_THRESHOLD = 0.20  #@param {type:"number"}
WHISPER_AT_REVIEW_THRESHOLD = 0.20  #@param {type:"number"}

tagger_review_df = verification_windows_df.copy()

if 'AST_RESULTS_DF' in globals():
    tagger_review_df = tagger_review_df.merge(AST_RESULTS_DF, on='window_index', how='left')
else:
    print('AST_RESULTS_DF not found; run the AST cell if you want AST scores included.')

if 'WHISPER_AT_RESULTS_DF' in globals():
    tagger_review_df = tagger_review_df.merge(WHISPER_AT_RESULTS_DF, on='window_index', how='left')
else:
    print('WHISPER_AT_RESULTS_DF not found; run the Whisper-AT cell if you want Whisper-AT scores included.')

if 'ast_laughter_score' in tagger_review_df.columns:
    tagger_review_df['ast_laughter_flag'] = tagger_review_df['ast_laughter_score'] >= AST_REVIEW_THRESHOLD
if 'whisper_at_laughter_score_max' in tagger_review_df.columns:
    tagger_review_df['whisper_at_laughter_flag'] = tagger_review_df['whisper_at_laughter_score_max'] >= WHISPER_AT_REVIEW_THRESHOLD

flag_columns = [col for col in ['ast_laughter_flag', 'whisper_at_laughter_flag'] if col in tagger_review_df.columns]
if flag_columns:
    tagger_review_df['tagger_vote_count'] = tagger_review_df[flag_columns].fillna(False).astype(int).sum(axis=1)

TAGGER_REVIEW_CSV = OUTPUT_DIR / f'{AUDIO_PATH.stem}_omine_ast_whisper_at_review.csv'
TAGGER_REVIEW_JSON = OUTPUT_DIR / f'{AUDIO_PATH.stem}_omine_ast_whisper_at_review.json'
tagger_review_df.to_csv(TAGGER_REVIEW_CSV, index=False)
TAGGER_REVIEW_JSON.write_text(
    json.dumps(
        {
            'source_audio': str(AUDIO_PATH),
            'ast_review_threshold': AST_REVIEW_THRESHOLD,
            'whisper_at_review_threshold': WHISPER_AT_REVIEW_THRESHOLD,
            'rows': tagger_review_df.to_dict(orient='records'),
        },
        indent=2,
        ensure_ascii=False,
    ),
    encoding='utf-8',
)

print(f'Combined review CSV: {TAGGER_REVIEW_CSV}')
print(f'Combined review JSON: {TAGGER_REVIEW_JSON}')
display(tagger_review_df.head(30))


In [ ]:
#@title Plot laughter timeline

import matplotlib.pyplot as plt
import librosa

try:
    audio_duration = librosa.get_duration(path=str(AUDIO_PATH))
except TypeError:
    audio_duration = librosa.get_duration(filename=str(AUDIO_PATH))

fig, ax = plt.subplots(figsize=(14, 2.5))
if not df.empty:
    bars = [(row.start_sec, row.duration_sec) for row in df.itertuples()]
    ax.broken_barh(bars, (0, 1), facecolors='tab:blue')

ax.set_xlim(0, audio_duration)
ax.set_ylim(0, 1)
ax.set_yticks([])
ax.set_xlabel('Time (seconds)')
ax.set_title(f'Laughter timeline: {AUDIO_PATH.name}')
ax.grid(axis='x', alpha=0.25)
plt.show()


In [ ]:
#@title Optional: listen to exported Drive laughter clips

from pathlib import Path

from IPython.display import Audio, display

MAX_CLIPS_TO_DISPLAY = 20  #@param {type:"integer"}

CLIP_DIR = OUTPUT_DIR / f'{AUDIO_PATH.stem}_laughter_clips'

if 'df' in globals() and 'clip_path' in df.columns:
    clip_paths = [Path(path) for path in df['clip_path'].dropna().tolist()]
else:
    clip_paths = sorted(CLIP_DIR.glob('laugh_*.wav'))

clip_paths = [path for path in clip_paths if path.exists()]
print(f'Found {len(clip_paths)} exported clip(s) in {CLIP_DIR}')

for clip_path in clip_paths[:MAX_CLIPS_TO_DISPLAY]:
    print(clip_path.name)
    display(Audio(str(clip_path)))


## Practical calibration notes

The upstream inference script uses a 0.5 frame probability threshold, then merges events closer than 0.2 seconds and removes events shorter than 0.2 seconds. If you need a stricter or more sensitive detector, make a small manually reviewed validation set from your own audio, then tune those thresholds in a copy of `inference.py`.

For long recordings, review a sample of exported clips. Laughter segmentation models can confuse applause, breathy speech, crowd noise, or music with laughter depending on the recording style.
